# 슬라이드 01. Vidu S2

> **핵심:** Vidu S2는 실시간 생성, 실시간 편집, 공간 영상까지 하나의 흐름으로 확장한 시스템이다.

### 발표 멘트

오늘 소개할 논문은 **Vidu S2**입니다. 핵심 질문은 하나입니다. 고품질 영상을 미리 만들어 두는 것을 넘어, 사용자의 말과 행동에 반응하면서 영상을 실시간으로 계속 생성하고 편집할 수 있는가입니다.

Vidu S2는 이를 위해 **Avatar**, **Editing**, 그리고 **Spatial Video** 세 가지 기능을 제시합니다.

**다음 연결:** 먼저 왜 실시간 영상 생성이 필요한지부터 보겠습니다.



# 슬라이드 02. 발표 흐름

> **핵심:** 논문의 순서를 그대로 따라 문제 제기부터 실험 결과까지 설명한다.

### 발표 멘트

발표는 여섯 단계로 진행하겠습니다. 먼저 실시간 영상 생성의 필요성과 이전 모델인 Vidu S1을 보고, 이어서 S2-Avatar의 데이터와 학습 방법을 설명하겠습니다.

그다음 실시간 편집 모델, 공간 영상 확장, 실험 결과를 본 뒤 결론으로 마무리하겠습니다.

### 화면에서 짚기

- 1장 Introduction
- 2장 Vidu S2-Avatar
- 3장 Vidu S2-Editing
- 4장 Spatial Video
- 5장 Experiments
- 6장 Conclusion

**다음 연결:** 먼저 오프라인 생성과 실시간 생성의 차이입니다.



# 슬라이드 03. 왜 실시간 영상 생성인가?

> **핵심:** 미리 완성하는 영상이 아니라, 사용자의 입력에 즉시 반응하는 영상 생성이 목표다.

### 발표 멘트

기존 영상 생성은 프롬프트를 넣고 수분을 기다린 뒤 완성된 영상을 받는 방식입니다. 영화나 광고처럼 미리 제작하는 작업에는 적합하지만, 대화·게임·라이브 방송처럼 사용자가 계속 개입하는 상황에는 맞지 않습니다.

저자들은 오프라인 영상은 한 번 만들어 여러 사람이 재생할 수 있지만, 실시간 콘텐츠는 사용자마다 매 순간 새로 생성되어야 하므로 장기적인 생성 수요가 더 클 수 있다고 봅니다. 이 계산은 엄밀한 시장 분석이라기보다 연구 방향의 필요성을 강조하는 직관입니다.

**다음 연결:** 이 문제를 처음 다룬 기반이 Vidu S1입니다.



# 슬라이드 04. Vidu S1의 출발점

> **핵심:** S1은 음성 제어, 무한 스트리밍, 실시간 속도를 한 시스템에 결합했다.

### 발표 멘트

Vidu S1은 세 가지 기반을 만들었습니다. 첫째, 음성 지시로 이후 행동을 바꿀 수 있습니다. 둘째, 자기회귀 방식으로 영상 길이를 계속 늘릴 수 있습니다. 셋째, 생성 단계를 줄여 실시간 속도에 접근했습니다.

오른쪽 예시처럼 사람뿐 아니라 다양한 캐릭터를 지원한다는 점도 중요합니다. S2는 이 구조를 이어받아 720p, 더 큰 신체 동작, 실시간 편집으로 범위를 넓힙니다.

### 화면에서 짚기

- 위: 음성 지시에 따라 동작이 순서대로 바뀌는 예시
- 아래 왼쪽: 낮은 비용·무한 길이 스트리밍
- 아래 오른쪽: 다양한 캐릭터

**다음 연결:** 무한 길이를 가능하게 하는 첫 번째 장치가 sliding window입니다.



# 슬라이드 05. Sliding-window Streaming Inference

> **핵심:** 전체 과거 대신 고정된 창만 유지해 영상이 길어져도 단계당 계산량을 고정한다.

### 발표 멘트

과거 프레임을 전부 attention에 넣으면 영상이 길어질수록 메모리와 계산량이 계속 증가합니다. S1은 최초 참조 구간, 최근 세 개의 history chunk, 현재 생성 구간만 유지합니다.

창이 이동할 때 오래된 chunk를 버리기 때문에 10초를 만들든 10분을 만들든 한 번의 생성 단계에서 보는 문맥 크기는 거의 같습니다.

### 화면에서 짚기

- Reference는 캐릭터의 기준
- History는 최근 생성 결과
- Current는 지금 denoising 중인 구간

**다음 연결:** 다만 창을 옮기면 최초 캐릭터 정보와 위치 정보가 흔들릴 수 있습니다.



# 슬라이드 06. Persistent Reference와 RoPE Repositioning

> **핵심:** 참조 블록은 고정하고, 이동한 history의 위치 정보만 다시 맞춘다.

### 발표 멘트

오래된 프레임을 계속 제거하면 얼굴·헤어스타일·의상 같은 정체성 정보도 사라질 수 있습니다. 그래서 최초 참조 이미지와 첫 생성 블록은 **sink block**으로 남겨 계속 기준점으로 사용합니다.

또 창이 이동하면 chunk의 상대 위치가 바뀝니다. 이때 KV cache 전체를 다시 계산하지 않고, 기존 feature에 현재 창 기준의 RoPE 위치만 다시 적용합니다.

### 화면에서 짚기

- 위 창의 Chunk 8·9가 아래 창에서 앞으로 이동
- Reference는 두 창 모두에 남아 있음

**다음 연결:** 문맥을 관리한 다음에는 과거 정보를 어떤 품질로 저장할지가 중요합니다.



# 슬라이드 07. TwinCache

> **핵심:** 움직임에는 noisy cache, 외형 복원에는 clean cache를 사용한다.

### 발표 멘트

TwinCache는 과거 영상 chunk를 두 형태로 저장합니다. 중간 denoising 단계의 **noisy cache**는 큰 움직임과 시간 흐름을 전달하면서 작은 오류가 다음 구간에 복사되는 것을 줄입니다.

최종 단계의 **clean cache**는 얼굴, 머리, 의상처럼 선명한 외형 정보를 복원합니다. 즉, 움직임의 안정성과 세부 묘사를 서로 다른 cache가 담당합니다.

**다음 연결:** 이 추론 구조를 만들기 위해 학습도 세 단계로 진행합니다.



# 슬라이드 08. S1의 3-stage Training

> **핵심:** 고품질 양방향 모델을 인과 모델로 바꾸고, 적은 denoising step으로 증류한다.

### 발표 멘트

첫 단계의 bidirectional teacher는 과거와 미래를 모두 보며 고품질 생성 능력을 학습합니다. 두 번째 causal teacher는 과거만 보도록 바꿔 스트리밍 구조에 적응합니다.

마지막으로 DMD와 PCM을 사용해 많은 denoising step을 소수 단계로 줄입니다. DMD는 학생의 생성 분포를 교사에 맞추고, PCM은 인접 생성 단계의 의미와 구조가 안정적으로 유지되도록 돕습니다.

**다음 연결:** 좋은 모델만큼 중요한 것이 학습 데이터의 정제입니다.



# 슬라이드 09. S1 데이터 파이프라인

> **핵심:** 원본 영상에서 실시간 캐릭터 학습에 적합한 단일 인물·단일 장면 클립만 남긴다.

### 발표 멘트

원본 영상은 먼저 해상도, FPS, 오디오·비디오 완전성을 검사합니다. 이후 장면 전환이 없는 3초에서 60초 길이의 single-shot clip으로 자릅니다.

한 명의 인물이 적절한 크기로 나온 영상만 남기고 자막, 워터마크, 흔들림, 유해 콘텐츠를 제거합니다. 마지막으로 화자가 화면 속 인물인지 확인하는 diarization을 거쳐 캡션과 임베딩을 만듭니다.

### 화면에서 짚기

- 왼쪽에서 오른쪽으로 필터가 강해지며 데이터가 정제됨

**다음 연결:** 이 기반 위에서 S2-Avatar가 무엇을 확장했는지 보겠습니다.



# 슬라이드 10. Vidu S2-Avatar

> **핵심:** 말하는 얼굴을 넘어 큰 동작과 다양한 캐릭터를 실시간으로 제어한다.

### 발표 멘트

S2-Avatar는 참조 이미지와 사용자 지시를 받아 캐릭터 영상을 실시간으로 이어 생성합니다. S1보다 큰 변화는 춤처럼 팔·다리·몸통이 크게 움직이는 동작까지 instruction following 범위를 넓혔다는 점입니다.

또 사용 중에도 참조 대상을 바꿀 수 있어 캐릭터, 물체, 의상, 배경을 동적으로 갱신할 수 있습니다.

### 화면에서 짚기

- 왼쪽: 모바일 실시간 상호작용 예시
- 오른쪽: 다양한 캐릭터 선택과 생성 인터페이스

**다음 연결:** 이를 가능하게 한 S2의 데이터 준비 과정을 보겠습니다.



# 슬라이드 11. S2-Avatar Data Preparation

> **핵심:** 더 다양한 동작 데이터와 더 정확한 시간 캡션을 추가한다.

### 발표 멘트

S2는 talking head뿐 아니라 영화·TV, 솔로 댄스, 2D·3D 애니메이션까지 데이터 범위를 넓힙니다. 특히 춤 데이터가 큰 전신 동작을, 애니메이션 데이터가 비실사 캐릭터의 외형과 움직임을 보강합니다.

파이프라인은 clipping, filtering, speech processing, temporal dense captioning, embedding 순서입니다. S2에서는 VLM으로 미세한 편집점을 재검사하고, 실제 선명도가 높은 영상만 고르며, 완만한 카메라 움직임은 배경을 안정화해 활용합니다.

캡션도 고정 길이 구간이 아니라 행동이 바뀌는 event boundary를 기준으로 작성해, 어떤 동작이 언제 시작되고 끝나는지 모델이 알 수 있게 합니다.

### 화면에서 짚기

- Figure 2의 왼쪽 빨간 박스가 Avatar 데이터 파이프라인
- 아래 예시는 시간 구간별 행동 캡션

**다음 연결:** 이 데이터로 학습하는 전체 방법은 여섯 단계입니다.



# 슬라이드 12. S2-Avatar 전체 학습 과정

> **핵심:** 공동 생성 모델을 스트리밍에 적응시키고, 선호 정렬과 초해상도로 마무리한다.

### 발표 멘트

전체 과정은 여섯 단계입니다. 먼저 Audio-Visual Joint DiT로 영상과 오디오를 함께 생성하고, I2V와 R2V를 함께 학습합니다.

그다음 causal adaptation과 self-replay forcing으로 실제 스트리밍 오류에 적응합니다. 마지막으로 preference optimization으로 사람이 선호하는 결과에 맞추고, one-step refiner로 720p 해상도를 복원합니다.

### 화면에서 짚기

- 1~2: 기본 생성 능력
- 3~4: 스트리밍 안정성
- 5~6: 선호 품질과 해상도

**다음 연결:** 먼저 기본 모델의 입력과 출력을 수식으로 보겠습니다.



# 슬라이드 13. Audio-Visual Joint DiT

> **핵심:** 참조 이미지와 구간별 지시를 조건으로 영상·오디오 latent를 함께 복원한다.

### 발표 멘트

수식의 입력은 노이즈가 섞인 video-audio latent, diffusion timestep, 참조 이미지 r, 그리고 구간별 조건 c입니다. 출력은 깨끗한 video-audio latent의 예측값입니다.

중요한 점은 참조 이미지 r이 모든 구간에 공통으로 들어가 캐릭터의 정체성을 잡고, c는 구간마다 달라져 현재 수행할 행동을 지정한다는 것입니다.

### 화면에서 짚기

- r: 캐릭터 참조 이미지
- x: 구간별 video-audio latent
- c: 구간별 캡션 또는 사용자 지시

**다음 연결:** 참조 이미지를 쓰는 방식은 I2V와 R2V 두 가지입니다.



# 슬라이드 14. I2V: Image-to-Video

> **핵심:** 첫 프레임에서 자연스럽게 이어지는 영상 연속성을 학습한다.

### 발표 멘트

I2V에서는 참조 이미지가 목표 영상의 첫 프레임입니다. 따라서 모델은 같은 장면과 인물이 다음 프레임으로 자연스럽게 이어지도록 학습합니다.

이 학습은 첫 프레임과 생성 영상의 연결, 장면 연속성, 카메라 연속성을 강화합니다.

### 화면에서 짚기

- 왼쪽의 I2V 카드가 강조되어 있음

**다음 연결:** 하지만 같은 캐릭터를 전혀 다른 포즈와 장면에 놓으려면 R2V가 필요합니다.



# 슬라이드 15. R2V: Reference-to-Video

> **핵심:** 별도 참조 이미지에서 정체성을 가져와 다양한 포즈와 장면에 적용한다.

### 발표 멘트

R2V의 참조 이미지는 목표 영상의 첫 프레임일 필요가 없습니다. 캐릭터의 얼굴, 머리, 의상 같은 정체성을 알려주는 별도 이미지입니다.

그래서 동일 인물을 다른 포즈와 장면에서도 유지할 수 있습니다. S2는 I2V와 R2V를 한 모델에서 함께 학습하고, 각 segment에 별도 조건을 주어 행동 지시를 더 정확히 따르게 합니다.

### 화면에서 짚기

- 오른쪽의 R2V 카드가 강조되어 있음

**다음 연결:** 이 양방향 모델을 실시간으로 바꾸는 단계가 causal adaptation입니다.



# 슬라이드 16. Hybrid Teacher Forcing과 Diffusion Forcing

> **핵심:** 안정적인 정답 과거와 불완전한 noisy 과거를 섞어 causal 모델을 학습한다.

### 발표 멘트

양방향 모델은 미래까지 보기 때문에 실시간에 사용할 수 없습니다. 그래서 현재 구간이 참조 이미지, 현재 지시, 현재 noisy segment, 유효한 과거만 보도록 causal attention으로 바꿉니다.

Teacher forcing은 깨끗한 정답 과거를 사용해 학습을 안정화합니다. Diffusion forcing은 과거에 서로 다른 수준의 노이즈를 넣어 불완전한 history에도 견디게 합니다.

하지만 둘 다 실제 모델이 직접 생성한 과거 분포를 완전히 경험하지 못한다는 한계가 남습니다.

**다음 연결:** 그 train-test gap을 줄이는 핵심이 Self-Replay Forcing입니다.



# 슬라이드 17. Self-Replay Forcing

> **핵심:** 모델이 실제로 만든 오류 궤적을 다시 재생해 장기 오류 누적을 학습한다.

### 발표 멘트

먼저 현재 student가 실제 추론 방식으로 긴 영상을 자기회귀 생성합니다. 생성된 block과 KV cache는 메모리를 줄이기 위해 계산 그래프에서 분리하고, 각 구간에 다시 노이즈를 넣습니다.

그다음 여러 구간을 하나의 causal pass로 replay하고 DMD supervision을 적용합니다. replay 내부에서는 gradient가 구간 사이로 흐르므로 뒤쪽에서 드러난 오류가 앞쪽 표현까지 교정할 수 있습니다.

결과적으로 얼굴·의상·배경 drift, 동작 반복, 장시간 collapse를 직접 학습 대상으로 삼습니다.

### 화면에서 짚기

- 슬라이드의 1→6 순서를 그대로 따라 설명

**다음 연결:** 안정성 다음에는 사람이 선호하는 품질로 정렬합니다.



# 슬라이드 18. Preference Optimization

> **핵심:** 양방향 단계와 스트리밍 단계에 서로 다른 선호 최적화를 적용한다.

### 발표 멘트

양방향 모델에는 diffusion DPO를 적용해 시각 품질, 표정, 동작 자연스러움, 오디오-비디오 동기화를 높입니다. 좋은 teacher를 만드는 단계입니다.

causal streaming 단계에는 Streaming NFT를 적용합니다. 정답 상태가 아니라 모델이 실제로 생성한 trajectory에서 보상을 주어 동작 제어와 지시 이행을 개선합니다.

### 화면에서 짚기

- 왼쪽: Bidirectional Stage - Diffusion DPO
- 오른쪽: Streaming Stage - Streaming NFT

**다음 연결:** 마지막으로 실시간 속도를 유지하면서 720p 화질을 만드는 refiner입니다.



# 슬라이드 19. One-Step Super-Resolution Refiner

> **핵심:** 저해상도에서 움직임을 만들고 한 번의 정제로 720p 세부를 복원한다.

### 발표 멘트

backbone이 처음부터 720p latent를 생성하면 계산량이 커집니다. 그래서 backbone은 저해상도에서 움직임과 시간 구조를 만들고, upsampling한 latent에 고정된 수준의 노이즈를 더한 뒤 refiner가 한 번에 고해상도로 정제합니다.

backbone은 high-noise cache로 장기 움직임을 안정화하고, refiner는 low-noise high-resolution cache로 얼굴·의상·질감을 복원합니다.

### 화면에서 짚기

- U는 latent-space upsampling
- t_r은 고정 refinement timestep
- 한 번의 refiner pass로 720p 생성

**다음 연결:** 모델 구조 외에도 실제 지연 시간을 줄이는 시스템 최적화가 필요합니다.



# 슬라이드 20. Inference Infrastructure

> **핵심:** attention, 양자화, kernel, 실행 그래프, GPU 병렬화를 함께 최적화한다.

### 발표 멘트

첫째, layer 민감도에 따라 SageAttention, SpargeAttention, sparse-linear attention을 선택합니다. 둘째, linear layer의 weight와 activation을 W8A8로 계산하되 block마다 scale을 둬 outlier 영향을 줄입니다.

셋째, 작은 연산을 kernel fusion으로 묶고, 반복되는 실행 순서는 CUDA Graph로 재생합니다. 마지막으로 긴 video-audio token을 여러 GPU에 나누고 통신 activation도 양자화합니다.

즉, 실시간성은 한 가지 기법이 아니라 모델 연산과 GPU 실행 전체를 함께 줄인 결과입니다.

**다음 연결:** 사용자의 자유로운 지시를 모델 입력으로 바꾸는 역할은 agent가 담당합니다.



# 슬라이드 21. Agentic System

> **핵심:** VLM agent가 지시를 구조화하고 생성 결과를 확인해 다음 프롬프트를 갱신한다.

### 발표 멘트

사용자는 텍스트, 음성, 초기 이미지, 추가 참조 이미지를 제공합니다. VLM agent는 이를 해석해 Avatar용 프롬프트를 만들고, 생성된 프레임을 다시 보며 행동이 완료됐는지 확인합니다.

새 지시가 들어와도 바뀌지 않은 속성은 계속 프롬프트에 유지합니다. 예를 들어 ‘컵을 집고 웃어’라면, 웃는 단계에서도 ‘컵을 들고 있음’을 명시해 물체가 사라지는 것을 막습니다.

### 화면에서 짚기

- 입력 → VLM Agent → Prompt → Avatar → Video Frames → Feedback의 폐루프

**다음 연결:** 이 agent가 참조 이미지와 상태 변화를 처리하는 실제 예시입니다.



# 슬라이드 22. Agentic Interaction Examples

> **핵심:** 물체·장면·액세서리의 변경과 그 이후 상태까지 지속적으로 관리한다.

### 발표 멘트

첫 번째는 파란 컵 같은 물체 참조를 받아 캐릭터는 유지하고 물체만 생성하거나 교체하는 예시입니다. 두 번째는 새 배경 이미지를 받아 캐릭터가 자연스럽게 장면을 이동하도록 만드는 예시입니다.

세 번째는 모자를 벗고 다시 쓰는 연속 행동입니다. agent는 ‘벗는다’라는 동작뿐 아니라 ‘손에 들고 있다’, ‘다시 착용했다’라는 결과 상태까지 기록해 다음 구간에 유지합니다.

### 화면에서 짚기

- (a) Reference Object Generation & Replacement
- (b) Reference Scene Transition
- (c) Taking Off and Putting On Accessories

**다음 연결:** 이제 생성이 아니라 입력 영상을 실시간으로 바꾸는 S2-Editing으로 넘어가겠습니다.



# 슬라이드 23. Vidu S2-Editing

> **핵심:** 입력 영상의 움직임을 유지하면서 스타일·의상·인물·배경을 실시간으로 바꾼다.

### 발표 멘트

S2-Editing은 들어오는 video stream을 실시간으로 편집합니다. 지원 작업은 style transfer, virtual try-on, subject replacement, background replacement 네 가지입니다.

핵심 난점은 프레임 하나를 예쁘게 바꾸는 것이 아니라, 원본의 포즈와 카메라 움직임을 보존하면서 바뀐 외형을 시간 전체에 일관되게 유지하는 것입니다.

**다음 연결:** 먼저 이런 편집 모델을 학습할 전후 영상 쌍을 어떻게 만드는지 보겠습니다.



# 슬라이드 24. S2-Editing Data Preparation

> **핵심:** 원본 움직임과 목표 외형을 분리해 고품질 편집 전·후 쌍을 합성한다.

### 발표 멘트

Avatar 파이프라인을 통과한 고품질 영상에 더 엄격한 필터링과 label balancing을 적용하고, 총 80만 개를 네 작업에 20만 개씩 겹치지 않게 배분합니다.

style transfer는 먼저 원본 영상에서 surface-normal video를 추출해 구조와 움직임을 얻고, 한 프레임을 reference로 사용해 원본을 복원하는 V2V 모델을 학습합니다. 그다음 reference만 stylized image로 바꾸면 움직임은 같고 외형만 바뀐 target video를 만들 수 있습니다.

나머지 작업은 여러 공개 편집 모델의 후보를 생성한 뒤 post-filtering과 비교 선택을 거쳐 가장 좋은 pair만 남깁니다.

### 화면에서 짚기

- Figure 2 오른쪽 빨간 박스
- Stage 1: reconstruction 학습
- Stage 2: paired editing data 생성

**다음 연결:** 이 데이터로 학습하는 편집 모델의 입력을 수식으로 보겠습니다.



# 슬라이드 25. S2-Editing Model Overview

> **핵심:** source는 움직임을, reference는 새 외형을, instruction은 편집 범위를 제공한다.

### 발표 멘트

편집 모델은 noisy target latent와 timestep 외에 source video, 선택적 reference image, 구간별 editing instruction을 받습니다. 출력은 깨끗한 edited video latent입니다.

역할을 나누면 source video는 포즈·움직임·카메라를 제공하고, reference image는 새 인물·의상·배경·스타일을 제공합니다. instruction은 무엇을 바꾸고 무엇을 보존할지 지정합니다.

source, target, reference는 시간·공간 좌표가 다르기 때문에 각각에 맞는 condition-specific RoPE를 사용합니다.

**다음 연결:** 학습은 먼저 양방향 편집을 익힌 뒤 실시간 causal 편집으로 전환합니다.



# 슬라이드 26. Bidirectional Editing → Causal Streaming

> **핵심:** 같은 시점의 source 움직임을 보존한 채 편집 모델을 스트리밍 구조로 바꾼다.

### 발표 멘트

bidirectional 단계에서는 target 전체를 보며 편집 결과의 외형과 시간적 일관성을 학습합니다. 이때 각 target frame은 같은 시점의 source frame만 직접 참조하는 frame-aligned attention을 사용합니다.

그래서 source의 포즈와 타이밍은 유지하면서 reference의 외형을 모든 target frame에 일관되게 적용할 수 있습니다. 이후 block-wise causal attention으로 바꾸고, Avatar와 같은 hybrid forcing과 self-replay forcing으로 장기 편집 drift를 줄입니다.

### 화면에서 짚기

- 왼쪽: Bidirectional Video Editing Training
- 오른쪽: Causal Streaming Training

**다음 연결:** 이 생성·편집 시스템은 좌안과 우안 영상을 만드는 공간 영상으로도 확장됩니다.



# 슬라이드 27. Real-Time Spatial Video

> **핵심:** Avatar와 Editing 결과를 좌·우 시점의 공간 영상으로 확장한다.

### 발표 멘트

공간 영상은 왼쪽 눈과 오른쪽 눈에 약간 다른 영상을 보여 주어 깊이감을 만듭니다. S2-Avatar의 단안 영상은 depth를 추정하고 수평 시차로 변환해 left view와 right view를 생성할 수 있습니다.

S2-Editing은 일반 단안 영상을 편집한 뒤 공간 영상으로 바꾸거나, 기존 좌·우 영상을 가로로 붙여 한 번에 편집한 뒤 다시 분리할 수 있습니다. 두 시점을 함께 처리하면 인물과 스타일의 좌우 일관성을 높일 수 있습니다.

### 화면에서 짚기

- 왼쪽: Avatar의 좌·우 시점 예시
- 오른쪽: Editing의 원본/편집 공간 영상 예시

**다음 연결:** 이제 정량·정성 실험에서 실제 성능을 확인하겠습니다.



# 슬라이드 28. Avatar 정량 평가

> **핵심:** Vidu S2-Avatar는 품질·동기화·지시 이행·장기 일관성에서 가장 높은 종합 성능을 보인다.

### 발표 멘트

StreamAV-Bench는 영상과 오디오 품질, lip sync, 지시 이행, 인물과 배경의 일관성을 함께 평가합니다. 표의 Vidu S2-Avatar는 보고된 모든 지표에서 가장 높은 값을 기록합니다.

특히 Subject Consistency는 0.998, Background Consistency는 0.993입니다. 실시간 모델의 핵심인 장시간 인물·배경 유지에서 강점을 보인다는 결과입니다.

다만 이 표는 논문이 선택한 benchmark와 비교군에 기반한 결과이므로, 실제 서비스 품질 전체를 보장한다기보다 해당 평가 조건에서의 우위를 보여준다고 해석해야 합니다.

### 화면에서 짚기

- 맨 아래 Vidu S2-Avatar 행
- SC 0.998
- BC 0.993

**다음 연결:** 편집 모델은 사람 선호도 비교에서도 우위를 보입니다.



# 슬라이드 29. Editing 인간 선호도 평가

> **핵심:** Vidu S2-Editing은 비교 모델보다 전반적으로 더 자주 선호된다.

### 발표 멘트

사람 평가자는 overall, video quality, consistency, semantic adherence를 비교했습니다. Decart-Lucy2.5와 비교하면 전체 선호도가 72.7%, XMax-X2.0과 비교하면 86.7%입니다.

특히 consistency와 semantic adherence에서도 파란색 비율이 크게 나타납니다. 즉, 한 프레임의 편집 품질뿐 아니라 시간에 따른 외형 유지와 지시 이행에서 더 좋은 평가를 받았습니다.

### 화면에서 짚기

- 파란색: Vidu S2-Editing 선호
- 노란색: 동률
- 분홍색: 비교 모델 선호

**다음 연결:** 정성 예시에서는 장시간 일관성이 어떻게 보이는지 확인할 수 있습니다.



# 슬라이드 30. Avatar 정성 비교

> **핵심:** Vidu S2는 긴 생성에서도 얼굴·의상·손 구조를 상대적으로 안정적으로 유지한다.

### 발표 멘트

왼쪽 두 열은 여성 캐릭터, 오른쪽 두 열은 남성 캐릭터의 시간적 일관성과 시각 안정성을 비교합니다. Vidu S2는 얼굴 정체성, 헤어스타일, 의상, 손 구조가 비교적 안정적으로 유지됩니다.

비교 모델에서는 시간이 지나며 얼굴과 신체 비율이 달라지거나, 머리·의상이 바뀌고 손가락이나 눈썹이 왜곡되는 사례가 보입니다. 이 예시는 self-replay와 cache 설계가 목표로 한 장기 drift 감소를 시각적으로 보여줍니다.

### 화면에서 짚기

- Temporal consistency 행: 시간에 따른 정체성 유지
- Visual stability 행: 손·얼굴·의상 왜곡 여부

**다음 연결:** 마지막으로 논문의 기여를 두 모델 중심으로 정리하겠습니다.



# 슬라이드 31. Conclusion

> **핵심:** Vidu S2는 실시간 캐릭터 생성과 실시간 영상 편집을 하나의 스트리밍 방향으로 확장한다.

### 발표 멘트

정리하면 Vidu S2-Avatar는 실시간 720p 생성, 동적으로 바꿀 수 있는 reference, 춤 같은 강한 instruction following을 지원합니다. Vidu S2-Editing은 style transfer, virtual try-on, character replacement, background replacement를 실시간 stream에서 수행합니다.

기술적으로는 causal streaming, self-replay forcing, cache 설계, 추론 최적화를 결합해 품질과 속도를 함께 다룹니다. 또한 spatial video 가능성까지 보여 주며 실시간 상호작용형 영상 생성의 범위를 넓혔습니다.

**다음 연결:** 이상으로 발표를 마치겠습니다.



# 슬라이드 32. 감사합니다

> **핵심:** 질문을 받는다.

### 발표 멘트

이상으로 Vidu S2 논문 리뷰를 마치겠습니다. 감사합니다. 질문 있으시면 받겠습니다.

